# Applying the classifier to the corpus

Classifies every sentence of every book and aggregates the labels into per-book counts, i.e. the format of `data/space_counts_per_book.csv`.

1. Download the corpus from https://doi.org/10.5281/zenodo.15714828 (`de-corp_txt/fiction`, one sentence per line).
2. The model is loaded from [`katrohrb/de-setting-classifier`](https://huggingface.co/katrohrb/de-setting-classifier); to use your own, point `MODEL` at the output of `train_classifier.ipynb`.

A GPU is strongly recommended: the full corpus has several million sentences.

In [ ]:
# !pip install transformers torch pandas

In [ ]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import pipeline

MODEL = "katrohrb/de-setting-classifier"
CORPUS = Path("path/to/de-corp_txt/fiction")
RESULTS = Path("../predictions")
RESULTS.mkdir(exist_ok=True)

LABELS = ["perceived_space", "action_space", "visual_space", "descriptive_space", "no_space"]

In [ ]:
pipe = pipeline("text-classification", model=MODEL,
                device=0 if torch.cuda.is_available() else -1, batch_size=64)

pipe("Er trat ans Fenster und sah hinaus auf den verschneiten Hof.")

In [ ]:
for txt in tqdm(sorted(CORPUS.glob("*.txt"))):
    out = RESULTS / f"{txt.stem}-result.json"
    if out.exists():
        continue
    sentences = txt.read_text(encoding="utf-8").splitlines()
    results = pipe(sentences, truncation=True)
    out.write_text(json.dumps(results), encoding="utf-8")

## Per-book counts

In [ ]:
rows = []
for res in RESULTS.glob("*-result.json"):
    labels = Counter(r["label"] for r in json.loads(res.read_text(encoding="utf-8")))
    rows.append({"filename": res.name.replace("-result.json", ".txt"),
                 **{label: labels[label] for label in LABELS},
                 "num_sents": sum(labels.values())})
counts = pd.DataFrame(rows)

meta_cols = ["author_last", "author_first", "title", "year", "genre", "filename"]
meta = pd.read_csv("../data/space_counts_per_book.csv")[meta_cols].drop_duplicates("filename")
books = meta.merge(counts, on="filename")
books.to_csv("../data/space_counts_per_book_rerun.csv", index=False)
books.head()